<a href="https://colab.research.google.com/github/kulkarnisachin07/ik-week-1-it-agent/blob/dev/agent.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import os
import json
from openai import OpenAI
import time, random

In [ ]:
#1 . Get the Open API Key
client = OpenAI(api_key=userdata.get('OPENAI_APIKEY'))


In [ ]:
#2 Implement tool 1 - Server Health
def get_server_health(server_id:str) -> str:
  """Returns CPU and Memory usage for a given server."""
  print(print(f"-> TOOL: Checking health for {server_id}..."))

  metrics = {
        # Scenario 1: High CPU (Needs Restart)
        "payment-server-01": {"cpu": "98%", "memory": "40%", "status": "Warning"},

        # Scenario 2: Healthy (No Action Needed)
        "db-node-02": {"cpu": "12%", "memory": "60%", "status": "Healthy"},

        # Scenario 3: High Memory Leak (Needs Restart or Escalation)
        "auth-service-03": {"cpu": "45%", "memory": "95%", "status": "Critical"},

        # Scenario 4: Network/Dependency Failure (Needs Escalation)
        "search-index-09": {"cpu": "10%", "memory": "15%", "status": "Error"},

        # Scenario 5: Completely Normal
        "frontend-node-04": {"cpu": "25%", "memory": "30%", "status": "Healthy"},
    }

  server = metrics.get(server_id,{"error": "Server not found. Check the ID."})

  return json.dumps(server)


In [ ]:
#3 fetch recent logs
def fetch_recent_logs(server_id : str, lines : int = 5)->str:
  """ Get recent logs based on server id """
  # Different logs for different servers to trigger different agent behaviors
  log_database = {

        "payment-server-01": [
            "[INFO] Request received /pay/v1",
            "[WARN] CPU threshold exceeded 90%",
            "[WARN] Thread pool exhaustion",
            "[CRITICAL] Process hung, not accepting new connections",
            "[ERROR] Timeout waiting for thread"
        ],
        "db-node-02": [
            "[INFO] Backup started",
            "[INFO] Backup completed successfully",
            "[INFO] User query executed in 12ms",
            "[INFO] Health check: OK",
            "[INFO] Replication sync active"
        ],
        "auth-service-03": [
            "[INFO] Token validated user_882",
            "[WARN] Garbage collection taking too long (>5s)",
            "[ERROR] java.lang.OutOfMemoryError: Java heap space",
            "[CRITICAL] Application crashing due to memory leak",
            "[INFO] Restarting context..."
        ],
        "search-index-09": [
            "[INFO] Indexing started",
            "[ERROR] Connection refused: elastic-cluster-main:9200",
            "[ERROR] Failed to write document ID 4432",
            "[CRITICAL] Dependency Unreachable: Search Engine is down",
            "[ERROR] Retrying in 30s..."
        ],
        "frontend-node-04": [
            "[INFO] GET /home 200 OK",
            "[INFO] GET /assets/logo.png 200 OK",
            "[INFO] GET /login 200 OK",
            "[INFO] GET /api/v1/status 200 OK",
            "[INFO] Health check passed"
        ]
    }

  # Default logs if server not found in specific list
  default_logs = ["[INFO] System stable", "[INFO] Heartbeat signal received"]

  logs = log_database.get(server_id, default_logs)
  return json.dumps({"logs": logs[:lines]})


In [ ]:
# --- TASK 1: Implement the Restart Tool ---
def restart_service(server_id: str) -> str:
    """ Restart the server
    ### TODO: Implement this function.
    1. Print a message saying "-> TOOL: Restarting service..."
    2. Return a JSON string confirming the restart was successful.
       Example return: '{"status": "success", "message": "Server restarted successfully"}'
    """


    print(f"Restarting server {server_id} ..... ")
    time.sleep(2)
    server_status = {"status": "success", "message": "Server restarted successfully"}
    return json.dumps(server_status)

In [ ]:
# --- TASK 2: Implement the Escalation Tool ---
def escalate_to_engineer(summary: str) -> str:
    """
    ### TODO: Implement this function.
    1. Print a message saying "-> TOOL: Escalating to human..."
    2. Return a JSON string confirming the ticket was created.
    """
    # [WRITE YOUR CODE HERE]

    print(f"TOOL: Escalating to human...")
    time.sleep(2)
    result =  {"message": "Ticket assigned successfully to L2 engineer"}
    return json.dumps(result)

In [ ]:
# Map functions for the agent execution loop
AVAILABLE_FUNCTIONS = {
    "get_server_health": get_server_health,
    "fetch_recent_logs": fetch_recent_logs,
    "restart_service": restart_service,
    "escalate_to_engineer": escalate_to_engineer,
}

In [ ]:
tools_schema = [
    {
        "type": "function",
        "function": {
            "name": "get_server_health",
            "description": "Checks the current CPU and memory usage of a specific server.",
            "parameters": {
                "type": "object",
                "properties": {
                    "server_id": {"type": "string", "description": "The ID of the server, e.g., 'payment-server-01'"}
                },
                "required": ["server_id"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "fetch_recent_logs",
            "description": "Retrieves the most recent log entries from a server to diagnose errors.",
            "parameters": {
                "type": "object",
                "properties": {
                    "server_id": {"type": "string", "description": "The ID of the server."},
                    "lines": {"type": "integer", "description": "Number of log lines to fetch."}
                },
                "required": ["server_id"]
            }
        }
    },
    # --- >>>> TASK 3: Define Schema for restart_service ---
    {
        "type": "function",
        "function": {
            "name": "restart_service",
            "description": "### TODO: Write a description for the AI",
            "parameters": {
                "type": "object",
                "properties": {
                    "server_id": {"type": "string", "description": "The ID of the server."}
                },
                "required": ["server_id"]
            }
        }
    },
    # --- >>>> TASK 4: Define Schema for escalate_to_engineer ---
    {
        "type": "function",
        "function": {
            "name": "escalate_to_engineer",
            "description": "Escalates the issue to a human engineer when automated fixes fail or the error is unknown.",
            "parameters": {
                "type": "object",
                "properties": {
                     "summary": {"type": "string", "description": "Issue summary"}
                },
                "required": ["summary"]
            }
        }
    }
]

In [ ]:
def run_it_agent(user_issue: str):
    print(f"\n--- New Incident: {user_issue} ---")

    messages = [
        {"role": "system", "content": "You are a Level 1 IT Responder. Investigate server issues. "
                                      "If CPU or Memory is > 90%, restart the service. If logs show critical dependency errors (like connection refused) that a restart won't fix, escalate to an engineer."},
        {"role": "user", "content": user_issue}
    ]

    while True:
        print("\n[AI Thinking...]")
        response = client.chat.completions.create(
            model="gpt-4.1-mini",
            messages=messages,
            tools=tools_schema,
            tool_choice="auto"
        )

        response_msg = response.choices[0].message
        messages.append(response_msg)

        if response_msg.tool_calls:
            for tool_call in response_msg.tool_calls:
                func_name = tool_call.function.name
                func_args = json.loads(tool_call.function.arguments)

                # Retrieve the actual python function based on name
                function_to_call = AVAILABLE_FUNCTIONS.get(func_name)

                if function_to_call:
                    # Execute the function
                    tool_output = function_to_call(**func_args)

                    # --- TASK 5: Append the result to messages ---
                    # The API needs to know which tool_call_id this output belongs to.
                    # Create the dictionary that represents the "tool" message.

                    messages.append({
                        "role": "tool",
                        "tool_call_id": tool_call.id, # What goes here?
                        "name": func_name,         # What goes here?
                        "content": tool_output       # What goes here?
                    })



        else:
            print(f"\n[FINAL RESPONSE]: {response_msg.content}")
            break

In [ ]:
# Scenario A: Should trigger a restart (CPU is 98%)
run_it_agent("The payment-server-01 is extremely slow and timing out.")


--- New Incident: The payment-server-01 is extremely slow and timing out. ---

[AI Thinking...]
-> TOOL: Checking health for payment-server-01...
None

[AI Thinking...]
Restarting server payment-server-01 ..... 

[AI Thinking...]

[FINAL RESPONSE]: The payment-server-01 had a CPU usage of 98%, which is very high, and the logs showed warnings and critical errors likely due to resource exhaustion. I have restarted the service to resolve the issue. Please monitor the server and let me know if the problem persists.


In [ ]:
# Scenario B: Should trigger an escalation (DB is healthy but logs might be weird)
run_it_agent("Something is wrong with db-node-02")


--- New Incident: Something is wrong with db-node-02 ---

[AI Thinking...]
-> TOOL: Checking health for db-node-02...
None

[AI Thinking...]

[FINAL RESPONSE]: The server db-node-02 is showing healthy CPU (12%) and memory (60%) usage. Recent logs do not show any critical errors or issues. Could you please specify what problem you are experiencing with db-node-02?


In [ ]:
# Scenario C: The High Memory Case (auth-service-03)
# Agent should see Memory 95% + OutOfMemoryError logs -> Restart
run_it_agent("Users are reporting login failures on auth-service-03.")

print("\n" + "="*50 + "\n")


--- New Incident: Users are reporting login failures on auth-service-03. ---

[AI Thinking...]
-> TOOL: Checking health for auth-service-03...
None

[AI Thinking...]
Restarting server auth-service-03 ..... 

[AI Thinking...]

[FINAL RESPONSE]: The server auth-service-03 was experiencing high memory usage at 95%, causing application crashes and login failures. I have restarted the service to address the issue. Please verify if the login issue is resolved now.




In [ ]:
# Scenario D: The Dependency Failure (search-index-09)
# Agent should see healthy CPU but "Connection Refused" logs -> Escalate
run_it_agent("Search isn't working. Can you check search-index-09?")

print("\n" + "="*50 + "\n")


--- New Incident: Search isn't working. Can you check search-index-09? ---

[AI Thinking...]
-> TOOL: Checking health for search-index-09...
None

[AI Thinking...]
TOOL: Escalating to human...

[AI Thinking...]

[FINAL RESPONSE]: The search-index-09 server is showing critical dependency errors related to connection refused to the elastic-cluster-main and the search engine being down. These errors won't be resolved by restarting the service. I have escalated the issue to an engineer for further investigation. If you need any more assistance, please let me know.




In [ ]:
# Scenario E: The Healthy Server (frontend-node-04)
# Agent should see normal stats and 200 OK logs -> Do nothing / Report healthy
run_it_agent("Check frontend-node-04 just to be safe.")


--- New Incident: Check frontend-node-04 just to be safe. ---

[AI Thinking...]
-> TOOL: Checking health for frontend-node-04...
None

[AI Thinking...]

[FINAL RESPONSE]: The server frontend-node-04 is healthy with CPU usage at 25% and memory usage at 30%. The recent logs do not show any critical errors. No further actions are needed at this time.
